# 🎬 RAG Studio: YouTube Summariser + Mind Map + Quiz Generator
Powered by **free Hugging Face models**, local embeddings (`all-MiniLM-L6-v2`) and a **Gradio** UI.

**Two ways to run the LLM (pick in the dropdown):**
- 🖥️ **`local:`** runs an open model *inside Colab* on the free GPU. No key, no quota. Set **Runtime → Change runtime type → T4 GPU** first.
- ☁️ **`api:`** calls Hugging Face Inference Providers (free monthly credits). Needs a free token: huggingface.co/settings/tokens, then add it in Colab 🔑 *Secrets* as `HF_TOKEN` (notebook access on).

Then **Runtime → Run all**. The last cell prints a public Gradio link.

**How it works (RAG):** transcript → chunks → embeddings → retrieval for Q&A; long videos are condensed in stages so they fit small-model context windows.

In [1]:
!pip -q install huggingface_hub youtube-transcript-api sentence-transformers gradio graphviz pypdf transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.0 MB/s eta 0:00:00


In [2]:
import os
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF token found ✅ (api: models available)" if HF_TOKEN else "No HF_TOKEN set. That's fine for local: models.")

HF token found ✅ (api: models available)


## 1. Core helpers (local + API model backends, retry)

In [3]:
import re, json, time, gc, textwrap
import numpy as np
import torch
from huggingface_hub import InferenceClient

MODELS = [
    "local: Qwen/Qwen2.5-3B-Instruct",
    "local: Qwen/Qwen2.5-7B-Instruct (4-bit)",
    "api: meta-llama/Llama-3.1-8B-Instruct",
    "api: Qwen/Qwen2.5-72B-Instruct",
]
HAS_GPU = torch.cuda.is_available()
DEFAULT_MODEL = MODELS[0] if HAS_GPU else MODELS[2]
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none (defaulting to api: model; switch runtime to T4 for local:)")
print("Default model:", DEFAULT_MODEL)

# ---------- local backend ----------
_LOCAL = {}
def _load_local(name, four_bit):
    global _LOCAL
    if _LOCAL.get("key") == (name, four_bit):
        return
    _LOCAL = {}; gc.collect(); torch.cuda.empty_cache()
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(name)
    kw = dict(device_map="auto")
    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16)
    else:
        kw["torch_dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(name, **kw)
    _LOCAL = {"key": (name, four_bit), "tok": tok, "model": model}

def _local_chat(messages, name, temperature, max_tokens):
    if not HAS_GPU:
        raise RuntimeError("No GPU. Use Runtime -> Change runtime type -> T4 GPU, or pick an api: model.")
    four_bit = "(4-bit)" in name
    name = name.replace("(4-bit)", "").strip()
    _load_local(name, four_bit)
    tok, model = _LOCAL["tok"], _LOCAL["model"]
    enc = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    gen = dict(max_new_tokens=max_tokens, pad_token_id=tok.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**enc, **gen)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# ---------- API backend ----------
def _api_chat(messages, name, temperature, max_tokens):
    if not HF_TOKEN:
        raise RuntimeError("api: models need an HF_TOKEN (Colab Secrets). Or pick a local: model.")
    c = InferenceClient(api_key=HF_TOKEN)   # provider="auto"
    r = c.chat_completion(messages=messages, model=name, temperature=max(temperature, 0.01), max_tokens=max_tokens)
    return r.choices[0].message.content

def llm(messages, model, temperature=0.3, max_tokens=1024, json_mode=False, retries=4):
    backend, name = [s.strip() for s in model.split(":", 1)] if ":" in model else ("api", model.strip())
    for attempt in range(retries):
        try:
            if backend == "local":
                return _local_chat(messages, name, temperature, max_tokens)
            return _api_chat(messages, name, temperature, max_tokens)
        except Exception as e:
            msg = str(e).lower()
            if "402" in msg or "credit" in msg:
                raise RuntimeError("Free API credits used up. Switch to a local: model (needs T4 GPU) or wait for the monthly reset.")
            if backend == "api" and any(x in msg for x in ("429", "503", "rate", "loading", "overloaded")) and attempt < retries - 1:
                time.sleep(15 * (attempt + 1))
                continue
            raise

def parse_json(s):
    s = re.sub(r"^```(?:json)?|```$", "", s.strip(), flags=re.M).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", s, re.S)
        if m:
            return json.loads(m.group(0))
        raise

def fmt_time(t):
    if t is None:
        return ""
    t = int(t); h, r = divmod(t, 3600); m, s = divmod(r, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

GPU: Tesla T4
Default model: local: Qwen/Qwen2.5-3B-Instruct


## 2. Ingestion: YouTube transcript / pasted text / file → chunks → embeddings

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # free, runs locally (Hugging Face model)
LANGS = ["en", "en-US", "en-GB", "hi"]

def video_id_from(url):
    m = re.search(r"(?:v=|youtu\.be/|shorts/|embed/|live/)([A-Za-z0-9_-]{11})", url)
    if m:
        return m.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{11}", url.strip()):
        return url.strip()
    raise ValueError("Couldn't find a video ID in that URL.")

def fetch_transcript(url):
    """Returns list of (start_seconds, text). Works with old and new youtube-transcript-api."""
    vid = video_id_from(url)
    try:
        api = YouTubeTranscriptApi()
        try:
            fetched = api.fetch(vid, languages=LANGS)
        except Exception as e:
            if "NoTranscriptFound" not in type(e).__name__:
                raise
            fetched = next(iter(api.list(vid))).fetch()   # any available language
        return [(s.start, s.text) for s in fetched]
    except AttributeError:
        raw = YouTubeTranscriptApi.get_transcript(vid, languages=LANGS)
        return [(s["start"], s["text"]) for s in raw]

def text_to_segments(text):
    sents = re.split(r"(?<=[.!?।])\s+|\n+", text)
    return [(None, s.strip()) for s in sents if s.strip()]

def read_file(f):
    path = f if isinstance(f, str) else f.name
    if path.lower().endswith(".pdf"):
        from pypdf import PdfReader
        return "\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    return open(path, encoding="utf-8", errors="ignore").read()

def chunk_segments(segs, max_chars=1200):
    chunks, cur, start, n = [], [], None, 0
    for t, txt in segs:
        txt = txt.replace("\n", " ")
        if start is None:
            start = t
        cur.append(txt); n += len(txt) + 1
        if n >= max_chars:
            chunks.append({"start": start, "text": " ".join(cur)})
            cur, start, n = [], None, 0
    if cur:
        chunks.append({"start": start, "text": " ".join(cur)})
    return chunks

def chunk_line(c):
    ts = f"[{fmt_time(c['start'])}] " if c["start"] is not None else ""
    return ts + c["text"]

def build_index(chunks):
    return np.asarray(embedder.encode([c["text"] for c in chunks],
                                      normalize_embeddings=True, show_progress_bar=False))

def retrieve(question, k=5):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    idx = np.argsort(-(STORE["emb"] @ q))[:k]
    return [STORE["chunks"][i] for i in sorted(idx)]

STORE = {}   # single-user Colab app: holds the currently loaded document

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 3. Features: summary (map-reduce), mind map, quiz, Q&A

In [5]:
def split_blocks(text, size=6000):
    blocks, cur, n = [], [], 0
    for line in text.split("\n"):
        cur.append(line); n += len(line) + 1
        if n >= size:
            blocks.append("\n".join(cur)); cur, n = [], 0
    if cur:
        blocks.append("\n".join(cur))
    return blocks

def condense(text, model, words=180):
    return llm([
        {"role": "system", "content": "You write dense, faithful study notes."},
        {"role": "user", "content": f"Condense this transcript excerpt into at most {words} words of bullet notes. "
                                    f"Keep concrete facts, names and numbers, and keep the [mm:ss] timestamps of key points.\n\n{text}"},
    ], model, max_tokens=500)

def get_notes(model, limit=10000):
    """Condensed notes of the whole document (cached). Short docs are used as-is."""
    if STORE.get("notes"):
        return STORE["notes"]
    notes = "\n".join(chunk_line(c) for c in STORE["chunks"])
    for _ in range(3):
        if len(notes) <= limit:
            break
        notes = "\n".join(condense(b, model) for b in split_blocks(notes))
    STORE["notes"] = notes
    return notes

def summarise(model):
    if not STORE.get("chunks"):
        return "⚠️ Load a video / text first."
    try:
        notes = get_notes(model)
        return llm([
            {"role": "system", "content": "You are an expert summariser. Use only the provided material."},
            {"role": "user", "content": "Write a summary in Markdown with these sections:\n"
                "## TL;DR (2-3 sentences)\n## Key points (6-10 bullets)\n"
                "## Chapters (timestamp - title; only if timestamps exist)\n## Key takeaways (3-5 bullets)\n\n"
                f"Material:\n{notes}"},
        ], model, max_tokens=1500)
    except Exception as e:
        return f"⚠️ {e}"

# ---------- Mind map ----------
import graphviz
COLORS = ["#4f46e5", "#0891b2", "#059669", "#d97706", "#dc2626"]

def tree_to_outline(node, depth=0):
    out = "  " * depth + ("# " if depth == 0 else "- ") + node["title"] + "\n"
    for ch in node.get("children", []):
        out += tree_to_outline(ch, depth + 1)
    return out

def render_mindmap(tree, path="mindmap"):
    g = graphviz.Digraph(format="png")
    g.attr(rankdir="LR", bgcolor="white", nodesep="0.25", ranksep="0.6", dpi="160")
    counter = [0]
    def wrap(s):
        return "\n".join(textwrap.wrap(s, 26)) or s
    def add(node, parent, depth):
        nid = f"n{counter[0]}"; counter[0] += 1
        color = COLORS[min(depth, len(COLORS) - 1)]
        g.node(nid, wrap(node["title"]), shape="box", style="rounded,filled",
               fillcolor=color if depth < 2 else "white", fontcolor="white" if depth < 2 else "black",
               color=color, fontname="Helvetica", fontsize="13" if depth else "16")
        if parent:
            g.edge(parent, nid, color=color)
        for ch in node.get("children", []):
            add(ch, nid, depth + 1)
    add(tree, None, 0)
    return g.render(path, cleanup=True)

def make_mindmap(model, depth):
    if not STORE.get("chunks"):
        return None, "⚠️ Load a video / text first."
    try:
        notes = get_notes(model)
        raw = llm([
            {"role": "system", "content": "You turn material into hierarchical mind maps. Reply with JSON only."},
            {"role": "user", "content": 'Return ONLY JSON of the form {"title": str, "children": [{"title": str, "children": [...]}]}. '
                f"Max depth {depth} levels below the root, 4-7 main branches, every label <= 6 words.\n\nMaterial:\n{notes}"},
        ], model, temperature=0.2, max_tokens=1500, json_mode=True)
        tree = parse_json(raw)
        outline = tree_to_outline(tree)
        try:
            img = render_mindmap(tree)
        except Exception as e:
            img = None
            outline = f"(Graph render failed: {e})\n\n" + outline
        return img, "```\n" + outline + "```"
    except Exception as e:
        return None, f"⚠️ {e}"

# ---------- Questionnaire ----------
QTYPES = {
    "Multiple choice": "multiple-choice with exactly 4 options (A-D) and one correct answer",
    "True / False": "true-or-false statements",
    "Short answer": "short-answer questions with a concise model answer",
}

def make_quiz(model, n, qtype, difficulty):
    if not STORE.get("chunks"):
        return "⚠️ Load a video / text first."
    try:
        notes = get_notes(model)
        raw = llm([
            {"role": "system", "content": "You write accurate quiz questions strictly from the given material. Reply with JSON only."},
            {"role": "user", "content": f"Write {int(n)} {difficulty.lower()} {QTYPES[qtype]} questions. Return ONLY JSON: "
                '{"questions": [{"question": str, "options": [str, ...] (empty list if not multiple choice), '
                '"answer": str, "explanation": str}]}\n\n' f"Material:\n{notes}"},
        ], model, temperature=0.4, max_tokens=2500, json_mode=True)
        STORE["quiz"] = parse_json(raw)["questions"]
        return show_quiz(False)
    except Exception as e:
        return f"⚠️ {e}"

def show_quiz(show_answers):
    qs = STORE.get("quiz")
    if not qs:
        return "Generate a quiz first."
    out = []
    for i, q in enumerate(qs, 1):
        out.append(f"**Q{i}. {q['question']}**\n")
        for j, o in enumerate(q.get("options") or []):
            o = re.sub(r"^[A-Da-d][).:]\s*", "", o)
            out.append(f"- {'ABCD'[j] if j < 4 else j+1}) {o}")
        if show_answers:
            out.append(f"\n✅ **Answer:** {q['answer']}  \n💡 {q.get('explanation', '')}")
        out.append("\n---\n")
    return "\n".join(out)

# ---------- RAG Q&A ----------
def ask(question, model):
    if not STORE.get("chunks"):
        return "⚠️ Load a video / text first.", ""
    if not question.strip():
        return "Type a question.", ""
    try:
        hits = retrieve(question, k=5)
        context = "\n\n".join(chunk_line(c) for c in hits)
        ans = llm([
            {"role": "system", "content": "Answer ONLY from the context. If the answer is not in the context, say so. "
                                          "Cite timestamps like [mm:ss] when the context has them."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ], model, max_tokens=700)
        src = "\n\n".join(f"**{('[' + fmt_time(c['start']) + ']') if c['start'] is not None else '•'}** {c['text'][:220]}…" for c in hits)
        return ans, src
    except Exception as e:
        return f"⚠️ {e}", ""

## 4. Gradio app

In [ ]:
import gradio as gr

def load_source(url, text, file, model):
    STORE.clear()
    try:
        if url and url.strip():
            segs, title = fetch_transcript(url), "YouTube " + video_id_from(url)
        elif text and text.strip():
            segs, title = text_to_segments(text), "Pasted text"
        elif file:
            segs, title = text_to_segments(read_file(file)), "Uploaded file"
        else:
            return "⚠️ Provide a YouTube link, paste text, or upload a file."
    except Exception as e:
        return (f"⚠️ Could not get the transcript: `{type(e).__name__}`. YouTube often blocks Colab's IPs. "
                "Workaround: open the video → *…more* → *Show transcript*, copy it, and paste it into the text box.")
    chunks = chunk_segments(segs)
    if not chunks:
        return "⚠️ No text found."
    STORE.update(chunks=chunks, emb=build_index(chunks), title=title)
    words = sum(len(c["text"].split()) for c in chunks)
    return f"✅ Loaded **{title}**: {words:,} words in {len(chunks)} chunks. Use the tabs below."

with gr.Blocks(title="RAG Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 RAG Studio\nSummarise a YouTube video, build a mind map, generate a quiz, or ask questions, all grounded in the transcript.")
    with gr.Row():
        with gr.Column(scale=3):
            url = gr.Textbox(label="YouTube URL", placeholder="https://www.youtube.com/watch?v=...")
            text = gr.Textbox(label="...or paste text / transcript", lines=4)
            file = gr.File(label="...or upload .txt / .pdf", file_types=[".txt", ".pdf", ".md"], type="filepath")
        with gr.Column(scale=1):
            model = gr.Dropdown(MODELS, value=DEFAULT_MODEL, label="Model", allow_custom_value=True)
            load_btn = gr.Button("📥 Load", variant="primary")
    status = gr.Markdown()
    load_btn.click(load_source, [url, text, file, model], status)

    with gr.Tabs():
        with gr.Tab("📝 Summary"):
            sum_btn = gr.Button("Summarise", variant="primary")
            sum_out = gr.Markdown()
            sum_btn.click(summarise, model, sum_out)
        with gr.Tab("🧠 Mind map"):
            depth = gr.Slider(2, 4, value=3, step=1, label="Depth")
            mm_btn = gr.Button("Generate mind map", variant="primary")
            mm_img = gr.Image(label="Mind map", type="filepath")
            mm_outline = gr.Markdown()
            mm_btn.click(make_mindmap, [model, depth], [mm_img, mm_outline])
        with gr.Tab("❓ Quiz"):
            with gr.Row():
                n = gr.Slider(3, 15, value=5, step=1, label="Questions")
                qtype = gr.Dropdown(list(QTYPES), value="Multiple choice", label="Type")
                diff = gr.Dropdown(["Easy", "Medium", "Hard"], value="Medium", label="Difficulty")
            q_btn = gr.Button("Generate quiz", variant="primary")
            reveal = gr.Checkbox(label="Show answers")
            q_out = gr.Markdown()
            q_btn.click(lambda *a: (False, make_quiz(*a)), [model, n, qtype, diff], [reveal, q_out])
            reveal.change(show_quiz, reveal, q_out)
        with gr.Tab("💬 Ask (RAG)"):
            question = gr.Textbox(label="Your question", placeholder="What does the speaker say about ...?")
            ask_btn = gr.Button("Ask", variant="primary")
            ans = gr.Markdown()
            with gr.Accordion("Retrieved passages", open=False):
                srcs = gr.Markdown()
            ask_btn.click(ask, [question, model], [ans, srcs])
            question.submit(ask, [question, model], [ans, srcs])

demo.launch(share=True, debug=True)

/tmp/ipykernel_1129/1426113778.py:24: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="RAG Studio", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://353114252a168a1c6b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (3 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]